# INTRODUCTION TO MACHINE LEARNING COURSE

# Final project : Advanced Predictive Modeling & Market Segmentation

Installation of the basic packages

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Phase 1 : Data Engineering

Data loading

In [ ]:
url = "https://raw.githubusercontent.com/gildasedenakpo-sketch/Data_set_home/refs/heads/main/Melbourne_housing_FULL.csv"
df = pd.read_csv(url)

Let’s display the head, the tail, and the dimensions of the dataset to check that it has been loaded correctly.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info() # to have general informations about the columns of the dataset

# Data Cleaning

Let’s identify the total number of missing values in each column

In [ ]:
df.isna().sum()

To determine which method to use for imputing missing values in the database, I decided to calculate the missingness correlation matrix between the variables that contain missing values. The goal is to find out whether the missing values follow a random or systematic process based on the correlations I obtain.

First, I create a binary variable for each variable:
* 1 if the value is missing
* 0 if the value is present

---


In [ ]:
# Selection of columns where major missing values need to be addressed
cols = ['Price', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude',
       'Longtitude']

In [ ]:
# Binary matrix for cols
missing_matrix = df[cols].isnull().astype(int)

In [ ]:
missing_matrix.head()

In [ ]:
missing_matrix.tail()

Let's define a function to calculate correlation and p_values

In [ ]:
from scipy.stats import pearsonr # for showing the significance level of each coefficient

In [ ]:
def cal_pvalues(df):
  cols = df.columns
  corr_matrix = df.corr()
  # create an empty matrix for p-values
  p_matrix = pd.DataFrame(np.ones((len(cols), len(cols))), columns = cols,
                          index = cols)
  for i in range(len(cols)):
    for j in range(len(cols)):
      if i!=j:
        # calculate Pearson correlation and p-value
        corr, p_value = pearsonr(df.iloc[:,i], df.iloc[:,j])
        p_matrix.iloc[i,j] = p_value
  return corr_matrix, p_matrix

Let's generate the missingness correlation matrix

In [ ]:
missing_corr, p_corr = cal_pvalues(missing_matrix)

Let's create a formatted string matrix with "*" for p<0.05

In [ ]:
star_matrix = missing_corr.round(2).astype(str)
for i in range (len(star_matrix)):
  for j in range (len(star_matrix)):
    if p_corr.iloc[i,j] < 0.05 and i!=j :
      star_matrix.iloc[i,j] = star_matrix.iloc[i,j] + "*"

In [ ]:
star_matrix

(*) indicates that the correlation coefficient obtained is significant at the 5% level.

Let's plot the heatmap of the nullity correlation

In [ ]:
plt.figure(figsize=(12, 10))
heatmap = sns.heatmap(missing_corr,
                      annot=star_matrix,
                      fmt="",
                      cmap='coolwarm',
                      center=0,
                      linewidths=0.5,
                      cbar_kws={'label': 'Correlation Coefficient'})

plt.title("Nullity Correlation Heatmap")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Decision Rule Used When Interpreting the Nullity Correlation Heatmap

* If the correlation between two variables is significant and greater than 0.7, the missingness is considered systematic.

* Otherwise, the missingness is considered random.

Based on this rule, the following conclusions can be drawn:

* Missing values in the variables Bathroom, Car, Bedroom2, and Landsize show systematic missingness. Therefore, a KNN Imputer will be used. These variables correspond to internal property characteristics.

* Missing values in the variables Longitude and Latitude also show systematic missingness. Therefore, a KNN Imputer will be used. These variables correspond to structural and temporal features.

* Missing values in the variables YearBuilt and BuildingArea show systematic missingness as well. Therefore, a KNN Imputer will be used. These variables correspond to location and environmental attributes.

* Missing values in the Price variable show random missingness. Therefore, mean imputation will be used if the distribution is symmetric, and median imputation will be used otherwise.

### Data Visualization
####  Distributions of some variables

In [ ]:
plt.figure(figsize=(18,12))

for i, col in enumerate(cols, 1):
    plt.subplot(3,3,i)
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f'Distribution of {col}')

plt.tight_layout()
plt.show()


The histogram shows that the distribution of the Price variable is asymmetric (skewed)

Let's now imput missing values

In [ ]:
from sklearn.impute import KNNImputer

col1 = ['Bathroom', 'Car', 'Bedroom2', 'Landsize']
col2 = [ 'Longtitude', 'Lattitude']
col3 = ['YearBuilt', 'BuildingArea']
imputer = KNNImputer(n_neighbors=5)
df[col1] = imputer.fit_transform(df[col1])
df[col2] = imputer.fit_transform(df[col2])
df[col3] = imputer.fit_transform(df[col3])

In [ ]:
median_price = df['Price'].median()
df['Price'] = df['Price'].fillna(median_price)

In [ ]:
df.isna().sum()

In [ ]:
df.shape

I then decided to remove the rows containing missing values in the variables Distance, Postcode, CouncilArea, Regionname, and Propertycount. This operation only removes three rows from the dataset, and therefore it is not expected to have any negative impact on the objectives of the project.

In [ ]:
columns_to_check = ['Distance', 'Postcode', 'CouncilArea', 'Regionname', 'Propertycount']
df.dropna(subset=columns_to_check, inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# Convert the Date column to date format and YearBuilt to integer format
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['YearBuilt'] = df['YearBuilt'].astype(int)

In [ ]:
df.head()

Descriptive statistics for numerical variables

In [ ]:
df.drop(columns=['Date', 'YearBuilt']).describe()

Descriptive statistics for categorial variables

In [ ]:
df.describe(include='object')

# Let's deal with outliers

In [ ]:
# Identify outliers
colms = ['Rooms',	'Price',	'Distance',	'Postcode',	'Bedroom2',	'Bathroom',
         'Car',	'Landsize',	'BuildingArea',	'Lattitude',	'Longtitude',
         'Propertycount']

for col in colms:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3-Q1
  lower_bound = Q1 - 1.5*IQR
  upper_bound = Q3 + 1.5*IQR
  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  print(col,":", len(outliers))

I implemented an outlier treatment method based on the Interquartile Range (IQR). For each numerical variable, the lower and upper bounds were calculated in order to identify outliers. Values below the lower bound were replaced with the minimum acceptable value (the lower bound), while values above the upper bound were replaced with the maximum acceptable value (the upper bound). This approach helps reduce the influence of extreme values while preserving all observations in the dataset.

In [ ]:
for col in colms:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  # Replace outliers below the lower bound with the lower bound
  df.loc[df[col] < lower_bound, col] = lower_bound

  # Replace outliers above the upper bound with the upper bound
  df.loc[df[col] > upper_bound, col] = upper_bound



Here, I want to ensure that the outliers have been properly replaced.

In [ ]:
for col in colms:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3-Q1
  lower_bound = Q1 - 1.5*IQR
  upper_bound = Q3 + 1.5*IQR
  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  print(col,":", len(outliers))

Here, I decided to standardize the numerical variables using StandardScaler.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[colms] = scaler.fit_transform(df[colms])


In [ ]:
df.head()

# Phase 2 : Unsupervised feature discovery

# i. Dimensionality Reduction

## PCA
Principal component analysis will be performed on the numerical variables I previously named colms without the variable "price"

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_cols = [col for col in colms if col != 'Price']
x = df[pca_cols]

pca = PCA()
X_pca = pca.fit_transform(x)

In [ ]:
# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ration:", explained_variance_ratio)

In [ ]:
# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
print("Cumulative Explained Variance:", cumulative_explained_variance)

In [ ]:
# Plot explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.title('Cumulative Explained Variance by Principal Component')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.show()

To choose the number of components, we retain 95% of the variance.

In [ ]:
pca = PCA(n_components = 0.95)
pca_features = pca.fit_transform(x)
pca_df = pd.DataFrame(pca_features)
pca_df.columns = [f'PC{i+1}' for i in range(pca_df.shape[1])]

In [ ]:
pca_df.head()

These components represent new synthetic variables formed as linear combinations of the original numerical features, each oriented to capture a maximal and unique direction of variation in the data.
They capture 95% of the total variance in the original dataset

# t-SNE

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
y = df['Price']

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, max_iter= 500, random_state=42)
tsne_features = tsne.fit_transform(x)

In [ ]:
# Visualize t-SNE results
plt.figure(figsize=(10, 6))
plt.scatter(tsne_features[:, 0], tsne_features[:, 1], c=y, alpha=0.7, s=5)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Visualization of Features')
plt.colorbar()
plt.show()

In [ ]:
tsne_df = pd.DataFrame(tsne_features, columns=['TSNE_C1', 'TSNE_C2'])

In [ ]:
tsne_df.head()

TSNE_C1 and TSNE_C2 are simply the 2D coordinates produced by t‑SNE to place each data point in a reduced space. They are used to visualize proximity relationships and clusters that exist in the original high dimensional data.

# Autoencoders

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

Implement autoencoder

In [ ]:
# input dimension
input_dim = x.shape[1]

# latent space dimension
encoding_dim = 8

# input layer
input_layer = keras.Input(shape=(input_dim,))

# encoder
encoded = layers.Dense(64, activation="relu")(input_layer)
encoded = layers.Dense(32, activation="relu")(encoded)

# latent space
latent = layers.Dense(encoding_dim, activation="linear", name="latent")(encoded)

# decoder
decoded = layers.Dense(32, activation="relu")(latent)
decoded = layers.Dense(64, activation="relu")(decoded)

# reconstruction
output_layer = layers.Dense(input_dim, activation="linear")(decoded)

Generate models

In [ ]:
# autoencoder complet
autoencoder = keras.Model(inputs=input_layer, outputs=output_layer)

# encoder only (to extract the new features)
encoder = keras.Model(inputs=input_layer, outputs=latent)

autoencoder.summary()

Compilation

In [ ]:
autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

Training

In [ ]:
history_ae = autoencoder.fit(
    x,
    x,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    verbose=1
)

Loss curve

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_ae.history["loss"], label="train loss")
plt.plot(history_ae.history["val_loss"], label="validation loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Autoencoder Training Loss")
plt.legend()

plt.show()

Extract the new features

In [ ]:
features_ae = encoder.predict(x)
x_test_features_ae = encoder.predict(x)

In [ ]:
ae_columns = [f'AE_{i+1}' for i in range(features_ae.shape[1])]
ae_df = pd.DataFrame(features_ae,columns = ae_columns)

In [ ]:
ae_df.head()

The new variables produced by the autoencoder are the coordinates of the latent vector, meaning a compressed representation of each observation that captures the most essential information from the original data. They form an abstract encoding used to reconstruct the inputs or reveal underlying structures in the dataset

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
# MSE Global
mse_global = mean_squared_error(x, autoencoder.predict(x))
print("Global MSE:", mse_global)

In [ ]:
# MSE by features
mse_by_feature = np.mean(np.square(x - autoencoder.predict(x)))
print("MSE by feature:", mse_by_feature)

In [ ]:
# Original variance vs reconstructed
original_variance = x.var(axis=0)
reconstructed_variance = autoencoder.predict(x).var(axis=0)

In [ ]:
# Variance ratio retained per feature
variance_ratio = reconstructed_variance/ original_variance
print("Variance ratio retained per feature:", variance_ratio)

# Global average
print("Mean retained variance:", variance_ratio.mean())

99.09% of the information is preserved by the 8 new features generated by the autoencoder.

# ii. Clustering & Segmentation

Implement K-means clustering

In [ ]:
from sklearn.cluster import KMeans

Use elbow method to find optimal number of clusters

In [ ]:
inertia = []
K = range (1,21)
for k in K:
  kmeans = KMeans(n_clusters=k, random_state=42)
  kmeans.fit(x)
  inertia.append(kmeans.inertia_)


In [ ]:
# Let's plot Elbow method for optimal k

plt.figure(figsize = (8,6))
plt.plot(K, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow method for optimal k')
plt.show()

Based on this graphics, the optimal number of cluster is 5

Implement optimal kmeans algorithm

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(x)
clusters = kmeans.fit_predict(x)


In [ ]:
# Visualize results by taking account the 2 first columns
plt.figure (figsize=(10,6))
scatter = plt.scatter(x.iloc[:, 0], x.iloc[:, 1], c = clusters, cmap='viridis')
centers = kmeans.cluster_centers_

In [ ]:
# plot centroids by taking account the 2 first columns
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker = 'x',
            s = 200, linewidths = 3, label = 'Centroids')
plt.title('Segments identified by K-means')
plt.xlabel('Rooms')
plt.ylabel('Distance')
plt.legend()
plt.show()

Let's add the 'cluster number' column to the dataset

In [ ]:
df['cluster number'] = clusters

In [ ]:
df.head()

In [ ]:
df['cluster number']. value_counts()

Description of clusters, linking to real-world Melbourne market characteristics

In [ ]:
df.groupby("cluster number")["Price"].mean()

In [ ]:
df.groupby("cluster number")["Regionname"].value_counts(normalize=True)

In [ ]:
df.groupby("cluster number")["Type"].value_counts(normalize=True)

In [ ]:
df.groupby("cluster number")[["Rooms","Distance","Landsize"]].mean()

Characteristics of each clusters

# Cluster 0 : Small, low-priced properties close to the city center

* Price: low (-0.34)

* Rooms: very few (-0.91)

* Distance: close to the city center (-0.72)

* Landsize: very small (-1.27)

* Type: mix of houses and apartments (~45% each)

* Region: mainly Northern / Southern / Western Metropolitan

This cluster groups small and affordable urban properties (apartments or small houses close to the city center).

# Cluster 1 : Mid-range properties, mostly houses

* Price: slightly below average (-0.16)

* Rooms: slightly below average

* Distance: slightly closer to the center

* Landsize: slightly above average

* Type: mostly houses (63%)

* Region: Northern and Western Metropolitan

This cluster groups medium-sized houses in nearby suburban areas, with moderate prices.

# Cluster 2 : Large family houses

* Price: around the average (+0.04)

* Rooms: many (+1.13)

* Distance: slightly farther from the center (+0.30)

* Landsize: medium (+0.34)

* Type: almost exclusively houses (96%)

* Region: mainly Western and Northern Metropolitan

This cluster groups large family houses in suburban areas with average prices.

# Cluster 3 : Distant and relatively expensive neighborhoods

* Price: moderately high (+0.07)

* Rooms: close to the average

* Distance: far from the center (+0.95)

* Landsize: slightly large

* Type: mostly houses but relatively mixed

* Region: Southern / Eastern / South-Eastern Metropolitan

This cluster groups residential areas farther from the city center, with relatively large and somewhat expensive properties.

# Cluster 4 : Premium houses

* Price: very high (+0.78)

* Rooms: very numerous (+1.30)

* Distance: somewhat far from the center

* Landsize: large (+0.70)

* Type: almost exclusively houses (95%)

* Region: Southern and Eastern Metropolitan

This cluster groups large high-end houses located in desirable neighborhoods.

Let's add unsupervised features to initial dataset

In [ ]:
df = pd.concat([df.reset_index(drop=True), pca_df], axis=1)

In [ ]:
df = pd.concat([df.reset_index(drop=True), tsne_df], axis=1)

In [ ]:
df = pd.concat([df.reset_index(drop=True), ae_df], axis=1)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

# Phase 3 : The Model Tournament

# i. The Baseline

In [ ]:
from sklearn.model_selection import train_test_split


Let's split dataset into train and test dataset

In [ ]:
from sklearn.model_selection import train_test_split


# TARGET
y = df["Price"]


# FEATURES PHASE 2

PCA_features = [
    'PC1','PC2'
] # two first features of PCA

tsne_features = [
    'TSNE_C1','TSNE_C2'
]

autoencoder_features = [
    'AE_1','AE_2'
] # two first features of autoencoder

# NUMERIC DATA ONLY
df_num = df.select_dtypes(include=['number']).drop(columns=["YearBuilt"], errors='ignore')

# DATASETS

# Initial features
X_base = df_num.drop(columns=PCA_features + tsne_features + autoencoder_features + ["Price"], errors='ignore')

# Initial features + PCA dataset
X_pca = X_base.join(df_num[PCA_features])

# Initial features + Autoencoder dataset
X_ae = X_base.join(df_num[autoencoder_features])

# Initial features + TSNE dataset
X_tsne = X_base.join(df_num[tsne_features])

# Train / Split

## for intial features
Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

## for initial features + pca features(same y_train and y_test)
Xp_train = X_pca.loc[Xb_train.index]
Xp_test  = X_pca.loc[Xb_test.index]

## for initial features + autoencoder features
Xae_train = X_ae.loc[Xb_train.index]
Xae_test  = X_ae.loc[Xb_test.index]

## for initial features + tsne features
Xtsne_train = X_tsne.loc[Xb_train.index]
Xtsne_test  = X_tsne.loc[Xb_test.index]

Implement Linear model regression



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(model, X_test, y_test): # just define a function to evaluate my model

    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    return rmse, mae, r2

Train a linear model on initial features

In [ ]:
lin = LinearRegression()

lin.fit(Xb_train, y_train)

rmse_base, mae_base, r2_base = evaluate_model(lin, Xb_test, y_test)

In [ ]:
print("RMSE:", rmse_base)
print("MAE:", mae_base)
print("R2:", r2_base)

Train a linear model on initial features + pca features

In [ ]:
lin.fit(Xp_train, y_train)

rmse_enh, mae_enh, r2_enh = evaluate_model(lin, Xp_test, y_test)

In [ ]:
print("RMSE:", rmse_enh)
print("MAE:", mae_enh)
print("R2:", r2_enh)

Train a linear model on intial features + autoencoder features

In [ ]:
lin.fit(Xae_train, y_train)

rmse_enh, mae_enh, r2_enh = evaluate_model(lin, Xae_test, y_test)

In [ ]:
print("RMSE:", rmse_enh)
print("MAE:", mae_enh)
print("R2:", r2_enh)

Train a linear model on initial features + tsne features

In [ ]:
lin.fit(Xtsne_train, y_train)

rmse_enh, mae_enh, r2_enh = evaluate_model(lin, Xtsne_test, y_test)

In [ ]:
print("RMSE:", rmse_enh)
print("MAE:", mae_enh)
print("R2:", r2_enh)

# ii. The Ensemble

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

Hyperparameter Search

In [ ]:
rf = RandomForestRegressor(random_state=42)

rf_params = {
    "n_estimators": [100, 150],
    "max_depth": [5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [2, 4]
}

rf_search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=5,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42
)

Random Forest on initial features

In [ ]:
rf_search.fit(Xb_train, y_train)

rf_best_base = rf_search.best_estimator_

In [ ]:
rf_rmse_base, rf_mae_base, rf_r2_base = evaluate_model(
    rf_best_base,
    Xb_test,
    y_test
)

In [ ]:
print("RMSE:", rf_rmse_base)
print("MAE:", rf_mae_base)
print("R2:", rf_r2_base)

Random forest on intial features + pca features

In [ ]:
rf_search.fit(Xp_train, y_train)

rf_best_pca = rf_search.best_estimator_


In [ ]:
rf_rmse_pca, rf_mae_pca, rf_r2_pca = evaluate_model(
    rf_best_pca,
    Xp_test,
    y_test
)

In [ ]:
print("RMSE:", rf_rmse_pca)
print("MAE:", rf_mae_pca)
print("R2:", rf_r2_pca)

Random forest on initial features + autoencoder features

In [ ]:
rf_search.fit(Xae_train, y_train)

rf_best_ae = rf_search.best_estimator_


In [ ]:
rf_rmse_ae, rf_mae_ae, rf_r2_ae = evaluate_model(
    rf_best_ae,
    Xae_test,
    y_test
)

In [ ]:
print("RMSE:", rf_rmse_ae)
print("MAE:", rf_mae_ae)
print("R2:", rf_r2_ae)

Random forest on initial features + tsne features

In [ ]:
rf_search.fit(Xtsne_train, y_train)

rf_best_tsne = rf_search.best_estimator_


In [ ]:
rf_rmse_tsne, rf_mae_tsne, rf_r2_tsne = evaluate_model(
    rf_best_tsne,
    Xtsne_test,
    y_test
)

In [ ]:
print("RMSE:", rf_rmse_tsne)
print("MAE:", rf_mae_tsne)
print("R2:", rf_r2_tsne)

# Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

Hyperparameter Search

In [ ]:
gbr = GradientBoostingRegressor(random_state=42)

gbr_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0]
}

gbr_search = RandomizedSearchCV(
    gbr,
    gbr_params,
    n_iter=5,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42
)

Gradient Boosting on intial features

In [ ]:
gbr_search.fit(Xb_train, y_train)

gbr_best_base = gbr_search.best_estimator_

gbr_rmse_base, gbr_mae_base, gbr_r2_base = evaluate_model(
    gbr_best_base,
    Xb_test,
    y_test
)

In [ ]:
print("RMSE:", gbr_rmse_base)
print("MAE:", gbr_mae_base)
print("R2:", gbr_r2_base)

Gradient Boosting on intial features + pca features

In [ ]:
gbr_search.fit(Xp_train, y_train)

gbr_best_pca = gbr_search.best_estimator_

gbr_rmse_pca, gbr_mae_pca, gbr_r2_pca = evaluate_model(
    gbr_best_pca,
    Xp_test,
    y_test
)

In [ ]:
print("RMSE:", gbr_rmse_pca)
print("MAE:", gbr_mae_pca)
print("R2:", gbr_r2_pca)

Gradient Boosting on initial features + autoencoder features

In [ ]:
gbr_search.fit(Xae_train, y_train)

gbr_best_ae = gbr_search.best_estimator_

gbr_rmse_ae, gbr_mae_ae, gbr_r2_ae = evaluate_model(
    gbr_best_ae,
    Xae_test,
    y_test
)

In [ ]:
print("RMSE:", gbr_rmse_ae)
print("MAE:", gbr_mae_ae)
print("R2:", gbr_r2_ae)

Gradient Boosting on initial features + tsne features

In [ ]:
gbr_search.fit(Xtsne_train, y_train)

gbr_best_tsne = gbr_search.best_estimator_

gbr_rmse_tsne, gbr_mae_tsne, gbr_r2_tsne = evaluate_model(
    gbr_best_tsne,
    Xtsne_test,
    y_test
)

In [ ]:
print("RMSE:", gbr_rmse_tsne)
print("MAE:", gbr_mae_tsne)
print("R2:", gbr_r2_tsne)

# iii. The Deep Learner

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.callbacks import EarlyStopping

Implement Neural Network model

In [ ]:

def build_nn(input_dim):

    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    return model

Neural network on initial features

In [ ]:
nn_base = build_nn(Xb_train.shape[1])

history_base = nn_base.fit(
    Xb_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(patience=10)],
    verbose=0
)

pred = nn_base.predict(Xb_test)

nn_rmse_base = np.sqrt(mean_squared_error(y_test, pred))
nn_mae_base = mean_absolute_error(y_test, pred)
nn_r2_base = r2_score(y_test, pred)

In [ ]:
print("RMSE:", nn_rmse_base)
print("MAE:", nn_mae_base)
print("R2:", nn_r2_base)

Neural network on initial features + PCA features

In [ ]:
nn_pca = build_nn(Xp_train.shape[1])

history_pca = nn_pca.fit(
    Xp_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(patience=10)],
    verbose=0
)

pred = nn_pca.predict(Xp_test)

nn_rmse_pca = np.sqrt(mean_squared_error(y_test, pred))
nn_mae_pca = mean_absolute_error(y_test, pred)
nn_r2_pca = r2_score(y_test, pred)

In [ ]:
print("RMSE:", nn_rmse_pca)
print("MAE:", nn_mae_pca)
print("R2:", nn_r2_pca)

Neural network on initial features + autoencoder features

In [ ]:
nn_ae = build_nn(Xae_train.shape[1])

history_ae = nn_ae.fit(
    Xae_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(patience=10)],
    verbose=0
)

pred = nn_ae.predict(Xae_test)

nn_rmse_ae = np.sqrt(mean_squared_error(y_test, pred))
nn_mae_ae = mean_absolute_error(y_test, pred)
nn_r2_ae = r2_score(y_test, pred)

In [ ]:
print("RMSE:", nn_rmse_ae)
print("MAE:", nn_mae_ae)
print("R2:", nn_r2_ae)

Neural network on initial features + tsne features

In [ ]:
nn_tsne = build_nn(Xtsne_train.shape[1])

history_tsne = nn_tsne.fit(
    Xtsne_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(patience=10)],
    verbose=0
)

pred = nn_tsne.predict(Xtsne_test)

nn_rmse_tsne = np.sqrt(mean_squared_error(y_test, pred))
nn_mae_tsne = mean_absolute_error(y_test, pred)
nn_r2_tsne = r2_score(y_test, pred)

In [ ]:
print("RMSE:", nn_rmse_tsne)
print("MAE:", nn_mae_tsne)
print("R2:", nn_r2_tsne)

Graph of loss (initial features)

In [ ]:
plt.plot(history_base.history['loss'])
plt.plot(history_base.history['val_loss'])
plt.title("Training vs Validation Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])
plt.show()

Graph of loss (initial features + pca features)

In [ ]:
plt.plot(history_pca.history['loss'])
plt.plot(history_pca.history['val_loss'])
plt.title("Training vs Validation Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])
plt.show()

Graph of loss (initial features + autoencoder features)

In [ ]:
plt.plot(history_ae.history['loss'])
plt.plot(history_ae.history['val_loss'])
plt.title("Training vs Validation Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])
plt.show()

Graph of loss (initial features + tsne features)

In [ ]:
plt.plot(history_tsne.history['loss'])
plt.plot(history_tsne.history['val_loss'])
plt.title("Training vs Validation Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])
plt.show()

Overall, the Random Forest, Gradient Boosting, and Neural Network models outperformed the simple Linear Regression model. In addition, their performance improved with the inclusion of features generated by dimensionality reduction techniques (PCA, autoencoder, t-SNE). Their performances are quite similar

# Phase 4 : Critical Analysis

## Performance Audit

# 1)

The neural network does not provide any benefit that would justify its additional complexity. Although its performance is acceptable, it remains slightly below Gradient Boosting and roughly on par with Random Forest, while requiring:

* more extensive preprocessing (especially feature scaling),
*  longer training time,
* more delicate and sensitive hyperparameter tuning.

Signs of overfitting were also observed, highlighting its vulnerability to training instability. Since ensemble models achieve equal or better performance with lower complexity and greater robustness, using Deep Learning is not warranted in this context.

# 2)

The features generated through PCA, the autoencoder, and t‑SNE improved predictive performance. The variations observed in RMSE and R² indicate a real gain.
This suggests that the dimensionality‑reduction techniques captured additional useful information.

# 3)

Gradient Boosting appears to be the best candidate for production deployment.
It offers:

* the strongest performance across all scenarios,

* good robustness,

* an appealing trade-off between accuracy and interpretability,

* lower sensitivity to preprocessing compared to neural networks.

Its stability and efficiency make it a reliable solution for real-world applications.

# Ethical Review

# 1)

Variables generated by dimensionality reduction techniques such as PCA or t‑SNE can act as proxies for sensitive features.
For example, they may implicitly encode information related to geographic location, which is often correlated with socio‑economic status or demographic characteristics.
The model can then indirectly learn patterns associated with protected attributes, even if these attributes are not explicitly present in the data.

# 2)

In the context of automated real estate valuation (mortgage lending, insurance, taxation, etc.), the model could pose significant risks:

* indirect discrimination against certain neighborhoods or population groups,

* reinforcement of existing social or economic inequalities,

* lack of transparency in automated decisions.

These issues could affect access to financial services or lead to unfair pricing, especially if the model exploits biases present in historical data.

# 3)

Before any deployment, several safeguards should be implemented:

* conduct bias audits to evaluate performance across different subgroups,

* use interpretability tools (feature importance, SHAP, etc.) to analyze model behavior,

* review and, if necessary, remove variables that may act as proxies for sensitive attributes,

* incorporate human oversight in critical decisions,

regularly monitor the model’s performance and potential deviations in production.